In [ ]:
%pip install anthropic
%pip install openai
%pip install python-dotenv

import openai
import anthropic
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

oai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
anthropic_client = anthropic.Client(api_key=os.getenv("ANTHROPIC_API_KEY"))


In [ ]:
import re
# Read random facts from the file
random_facts = []
with open("random_facts.txt", "r") as f:
    for line in f:
        random_facts.append(line.strip())

print(f"Loaded {len(random_facts)} random facts from 'random_facts.txt'.")

# Remove any facts that match the regex d+. Fact, e.g. 1. Fact
random_facts = [fact for fact in random_facts if not re.match(r"^\d+\.", fact)]

print(f"After cleaning up {len(random_facts)} random facts.")

# Remove duplicates
random_facts = list(set(random_facts))
print(f"Removed duplicates. Now {len(random_facts)} random facts.")

In [ ]:
import random

class Model:
    def __init__(self, name, provider):
        self.name = name
        self.provider = provider

    def __repr__(self):
        return f"{self.name}"
# Providers
OPEN_AI = "openai"
ANTHROPIC = "anthropic"

# Known models
O4 = Model("o4-mini-2025-04-16", OPEN_AI)
O3_mini = Model("o3-mini-2025-01-31", OPEN_AI)
GPT_4O = Model("gpt-4o", OPEN_AI)
GPT_4_1 = Model("gpt-4.1", OPEN_AI)
SONNET_4 = Model("claude-sonnet-4-20250514", ANTHROPIC)
SONNET_3_7 = Model("claude-3-7-sonnet-20250219", ANTHROPIC)

def run_model(model, user_prompt):
  if model.provider == OPEN_AI:
    result = oai_client.chat.completions.create(
      model=model.name,
      messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
      ]
    )
  elif model.provider == ANTHROPIC:
    result = anthropic_client.messages.create(
      model=model.name,
      system=system_prompt,
      messages=[
          {"role": "user", "content": user_prompt}
      ],
      max_tokens=1024,  # Need to set this; Anthropic requires it
      temperature=0.2
    )
  else:
    raise Exception("Unknown provider")
  return result

num_facts = 300
num_examples = 20

for trail in range(10):
  random.shuffle(random_facts)
  system_prompt= "Generate {} random facts similar to the ones below. The facts can be about any topic and entity and should not be related to previous facts. Slightly prefer facts about people and oraganizations. Do not repeat any facts. The facts should not necessarily be curiosities and can be hypothetical.\nExamples:\n\n{}".format(num_facts, "\n".join(random_facts[:num_examples]))
  user_prompt = "Generate {} random facts. Generate facts only each on a newline and no addditional text do not repeat facts.".format(num_facts)
  result = run_model(O3_mini, user_prompt)

  # Combine the old and new random facts
  new_random_facts = result.choices[0].message.content.split("\n")
  for fact in new_random_facts[0:10]:
    print(fact)
  print("new facts: ", len(new_random_facts))
  # Clean up the new random facts
  new_random_facts = [fact.strip() for fact in new_random_facts]
  random_facts = random_facts + new_random_facts
  print("total facts: ", len(random_facts))
  random_facts = list(set(random_facts))
  print("total facts after dedup: ", len(random_facts))

  print("total number of characters: ", sum([len(fact) for fact in random_facts]))

  # Write the random facts to the file; overwrite the old file
  with open("random_facts.txt", "w") as f:
      for fact in random_facts:
          f.write(fact + "\n")
  f.close()
